<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [2]:
import sys
import os
sys.path.append(os.path.abspath("../../.."))

from config.spark_config import SparkConfig
from utils.logger import LoggerFactory
from config.io_config import *
from app.platform_app import PlatformApp
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *
from pyspark.sql import functions as F

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [3]:
# Initialize shared logger (all logs in this run go to the same file: etl_<run_id>.log)
logger = LoggerFactory.setup_logger(name="ETL", log_dir=LOG_DIR)

# Create Spark session with logging enabled (for tracing Spark-related operations)
spark = SparkConfig.create_spark(app_name="Paypal Analytic", logger=logger, use_databricks=True)

# Initialize main application with Spark and logger (used across ETL pipeline)
app = PlatformApp(spark=spark, logger=logger, catalog_name="paypal_analytic")

2026-04-07 23:41:54 | INFO     | ETL | logger.py:113 | Logger initialized | level=DEBUG | file=C:/01_Data/05-data-engineer-bootcamp/03_paypal_databricks/logs\etl_20260401_193104_713773.log
2026-04-07 23:41:56 | INFO     | ETL | spark_config.py:89 | Connected to Databricks via Spark Connect.
2026-04-07 23:41:56 | INFO     | ETL | platform_app.py:44 | Initializing Data Platform...
2026-04-07 23:41:56 | INFO     | ETL | platform_app.py:50 | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [4]:
df_bronze_transactions = spark.sql(f"SELECT * FROM {BRONZE_TRANSACTIONS}")

# Preview result
df_bronze_transactions.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Transformations

In [5]:
# check null
check_null(df = df_bronze_transactions, logger=logger)

2026-04-07 23:42:00 | WARNING  | ETL | data_quality.py:65 | 
+---------------------+---------------+-----------+
|      Features       | Missing_Count | Missing_% |
+---------------------+---------------+-----------+
| transaction_subject |     4412      |   92.07   |
+---------------------+---------------+-----------+
2026-04-07 23:42:00 | WARNING  | ETL | data_quality.py:70 | Total missing values: 4,412 out of 4,792 rows.


### Trim spaces

In [6]:
# Remove those trim values
df_bronze_transactions = clean_dataframe(df=df_bronze_transactions)

# Preview result
df_bronze_transactions.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Select Features

In [7]:
df_silver_transactions = df_bronze_transactions.select("transaction_id", "transaction_event_code", "transaction_initiation_date",
                                                       "transaction_updated_date", "transaction_amount", "ending_balance",
                                                       "available_balance", "transaction_status", "transaction_subject",
                                                       "protection_eligibility", "elton_created_at", "dt", "hour")

# Preview result
df_silver_transactions.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+----------------------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------+------------------------------+----------+----+
|transaction_id   |transaction_event_code|transaction_initiation_date   |transaction_updated_date      |transaction_amount                            |ending_balance                                |available_balance                             |transaction_status|transaction_subject                                                                                                            |protection_eligibility|elton_created_at              |dt        |hour|
+-----------------+----------------------+------------------------------+-

### Extract Data

In [8]:
# Select and transform transaction data
df_silver_transactions = df_silver_transactions.select(
    "transaction_id",
    "transaction_event_code",

    # Parse timestamps
    parse_timestamp(F.col("transaction_initiation_date")).alias("transaction_initiation_date"),
    parse_timestamp(F.col("transaction_updated_date")).alias("transaction_updated_date"),

    # Extract transaction amount (currency + absolute value)
    parse_decimal(parse_json_field(F.col("transaction_amount"), "$.value")).alias("transaction_amount"),

    # Extract ending balance
    parse_decimal(parse_json_field(F.col("ending_balance"), "$.value")).alias("ending_balance"),

    # Extract available balance
    parse_decimal(parse_json_field(F.col("available_balance"), "$.value")).alias("available_balance"),
    parse_json_field(F.col("available_balance"), "$.currency_code").alias("currency_code"),

    # transaction_status: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("transaction_status")) == "", None)
         .otherwise(F.col("transaction_status")),
        F.lit("Unknown")
    ).alias("transaction_status"),

    # transaction_subject: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("transaction_subject")) == "", None)
         .otherwise(F.col("transaction_subject")),
        F.lit("Unknown")
    ).alias("transaction_subject"),

    # Cast eligibility flag to integer
    F.col("protection_eligibility").cast("int").alias("protection_eligibility"),

    # Metadata fields
    parse_timestamp(F.col("elton_created_at")).alias("elton_created_at"),
    F.col("dt").cast("date").alias("dt"),
    F.col("hour").cast("int").alias("hour")
) \
.filter(F.col("transaction_id").isNotNull()) \
.withColumn("process_timestamp", F.date_trunc("second", F.current_timestamp()))

# Preview result
df_silver_transactions.show(n=10, truncate=False)

+-----------------+----------------------+---------------------------+------------------------+------------------+--------------+-----------------+-------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------+-------------------+----------+----+-------------------+
|transaction_id   |transaction_event_code|transaction_initiation_date|transaction_updated_date|transaction_amount|ending_balance|available_balance|currency_code|transaction_status|transaction_subject                                                                                                            |protection_eligibility|elton_created_at   |dt        |hour|process_timestamp  |
+-----------------+----------------------+---------------------------+------------------------+------------------+--------------+-----------------+-------------+------------------+--------------------------------------------

In [9]:
df_silver_transactions.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_event_code: string (nullable = true)
 |-- transaction_initiation_date: timestamp (nullable = true)
 |-- transaction_updated_date: timestamp (nullable = true)
 |-- transaction_amount: decimal(18,2) (nullable = true)
 |-- ending_balance: decimal(18,2) (nullable = true)
 |-- available_balance: decimal(18,2) (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- transaction_status: string (nullable = false)
 |-- transaction_subject: string (nullable = false)
 |-- protection_eligibility: integer (nullable = true)
 |-- elton_created_at: timestamp (nullable = true)
 |-- dt: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- process_timestamp: timestamp (nullable = true)



### Duplicates

In [10]:
df_silver_transactions = dedup(
    df_silver_transactions,
    dedup_cols=["transaction_id", "transaction_event_code"],
    order_cols=["transaction_updated_date", "dt", "hour", "elton_created_at"],
    logger=logger
)

2026-04-07 23:42:05 | INFO     | ETL | utils.py:156 | Starting deduplication
2026-04-07 23:42:05 | INFO     | ETL | utils.py:157 | Dedup columns: ['transaction_id', 'transaction_event_code']
2026-04-07 23:42:05 | INFO     | ETL | utils.py:158 | Order columns: ['transaction_updated_date', 'dt', 'hour', 'elton_created_at']
2026-04-07 23:42:05 | INFO     | ETL | utils.py:176 | Order direction (desc): [True, True, True, True]
2026-04-07 23:42:05 | INFO     | ETL | utils.py:177 | Nulls last: True
2026-04-07 23:42:06 | INFO     | ETL | utils.py:184 | Input row count: 4792
2026-04-07 23:42:06 | INFO     | ETL | utils.py:219 | Output row count after dedup: 4002
2026-04-07 23:42:06 | INFO     | ETL | utils.py:220 | Removed duplicate rows: 790
2026-04-07 23:42:06 | INFO     | ETL | utils.py:221 | Deduplication completed


### Transformed data to Silver Layer

In [11]:
if not spark.catalog.tableExists(SILVER_PATH_DISPUTED_PP01_TRANSACTIONS):
    logger.info("Silver disputed pp01 transactions table not found. Creating new table...")
    df_silver_transactions.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("append") \
                   .saveAsTable(SILVER_PATH_DISPUTED_PP01_TRANSACTIONS)
    logger.info("Silver disputed pp01 transactions table created successfully")
else:
    logger.info("Silver disputed pp01 transactions table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver_transactions, key_cols=["transaction_id", "transaction_event_code"],
           table=SILVER_TABLE_DISPUTED_PP01_TRANSACTIONS, cdc="transaction_updated_date",
           name_catalog=app.catalog_name, name_schema=SCHEMA_SILVER, logger=logger)
    logger.info("Upsert completed successfully")

2026-04-07 23:42:07 | INFO     | ETL | 2823656660.py:10 | Silver disputed pp01 transactions table exists. Performing upsert...
2026-04-07 23:42:07 | INFO     | ETL | utils.py:315 | Starting UPSERT into paypal_analytic.silver.disputed_pp01_transactions
2026-04-07 23:42:15 | INFO     | ETL | utils.py:345 | UPSERT completed successfully: paypal_analytic.silver.disputed_pp01_transactions
2026-04-07 23:42:15 | INFO     | ETL | 2823656660.py:14 | Upsert completed successfully


In [12]:
app.stop()

2026-04-07 23:42:15 | INFO     | ETL | platform_app.py:259 | Stopping Spark session...
2026-04-07 23:42:15 | INFO     | ETL | platform_app.py:261 | Spark stopped.
